# Gamma-prime instance segmentation: training and evaluation

This notebook documents the Mask R-CNN workflow used for the manuscript. 
The fixed COCO splits are provided in `data/segmentation/train.json` and `val.json` 
(88 training images, 23,603 annotations; 22 validation images, 6,702 annotations). 
The trained checkpoint is provided as `model/model_final.pth`. 
Paths should be adjusted to the local location of the SEM images before execution.


# **Mount Drive + set paths**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Install Detectron2**

In [ ]:
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available(), 'cuda ver', torch.version.cuda)"

# If CUDA is available but detectron2 not installed, install detectron2
try:
    import detectron2
    print("detectron2:", detectron2.__version__)
except Exception as e:
    print("Installing detectron2...")
    !pip -q install 'git+https://github.com/facebookresearch/detectron2.git'
    import detectron2
    print("detectron2 installed:", detectron2.__version__)

# **Merge old gamma'_ annotations + new CVAT export, then create train/val COCO splits**

# **Verify image files exist for the COCO split**

In [ ]:
def coco_list_files(coco_json):
    coco = json.load(open(coco_json, "r"))
    return [im["file_name"] for im in coco["images"]]

train_files = coco_list_files(SINGLE_TRAIN_JSON)
val_files   = coco_list_files(SINGLE_VAL_JSON)

missing_train = [f for f in train_files if not os.path.exists(os.path.join(IMG_DIR, f))]
missing_val   = [f for f in val_files   if not os.path.exists(os.path.join(IMG_DIR, f))]

print("Single-class TRAIN images:", len(train_files), "| missing:", len(missing_train))
print("Single-class VAL   images:", len(val_files),   "| missing:", len(missing_val))

if missing_train[:5]:
    print("Example missing train:", missing_train[:5])
if missing_val[:5]:
    print("Example missing val:", missing_val[:5])

assert len(missing_val) == 0, "VAL split has missing images in IMG_DIR. Fix IMG_DIR or COCO file_name."

# **Register datasets (Detectron2)**

In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog

DATASET_TRAIN = "gamma_prime_train"
DATASET_VAL   = "gamma_prime_val"

# If it is already registered, remove and re-register cleanly
for name in [DATASET_TRAIN, DATASET_VAL]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances(DATASET_TRAIN, {}, SINGLE_TRAIN_JSON, IMG_DIR)
register_coco_instances(DATASET_VAL,   {}, SINGLE_VAL_JSON,   IMG_DIR)

meta = MetadataCatalog.get(DATASET_TRAIN)
meta.thing_classes = ["gamma_prime"]

print("Registered datasets:", DatasetCatalog.list())
print("Train samples:", len(DatasetCatalog.get(DATASET_TRAIN)))
print("Val samples:", len(DatasetCatalog.get(DATASET_VAL)))

# **Ground Truth visualization (overlay) + save figure**

In [ ]:
import cv2
import matplotlib.pyplot as plt

from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import DatasetCatalog, MetadataCatalog

def show_gt_examples(dataset_name, n=6, save_path=None):
    dset = DatasetCatalog.get(dataset_name)
    meta = MetadataCatalog.get(dataset_name)

    picks = random.sample(dset, min(n, len(dset)))

    cols = 3
    rows = int(np.ceil(len(picks)/cols))
    plt.figure(figsize=(5*cols, 5*rows))

    for i, rec in enumerate(picks):
        img_path = rec["file_name"]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        v = Visualizer(img, metadata=meta, scale=1.0, instance_mode=ColorMode.IMAGE)
        out = v.draw_dataset_dict(rec)

        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(out.get_image())
        ax.set_title(f"{os.path.basename(img_path)} | GT inst={len(rec['annotations'])}")
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()

gt_save = os.path.join(FIG_DIR, "fig_gt_overlays_val.png")
show_gt_examples(DATASET_VAL, n=6, save_path=gt_save)

# **Train Mask R-CNN (γ′ 1-class) + save model**

### RAM-safe Colab training patch — V2

The first RAM patch exposed the real failure chain in the Colab log:
**CUDA OOM during `pairwise_iou` → Detectron2 CPU fallback → system RAM exhaustion**.

V2 therefore uses a **512×512 random training crop**, ResNet-50 FPN, batch size 1,
zero DataLoader workers, only 512 post-NMS training proposals, ROI batch size 128,
500 maximum test detections, and 512–768 validation/inference sizing. The later
deployment configuration is patched to the same lightweight R50 setup so it does
not switch back to the original R101/high-proposal configuration.


In [ ]:
import math
import os
import gc
import torch

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, hooks
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import build_detection_train_loader, DatasetMapper
from detectron2.data import transforms as T
from detectron2.evaluation import COCOEvaluator

# ============================================================
# COLAB T4 — EXTRA-SAFE dense γ′ Mask R-CNN configuration
#
# Why V2?
# The runtime log showed:
#   CUDA OOM in pairwise_iou
# followed by Detectron2 attempting the operation on CPU.
# With dense γ′ annotations that CPU fallback can then exhaust
# system RAM. This configuration reduces both GPU pressure and
# proposal × ground-truth matching size.
# ============================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

if DEVICE != "cuda":
    raise RuntimeError(
        "CUDA GPU is not active. In Colab choose "
        "Runtime > Change runtime type > T4 GPU, reconnect, then Run all."
    )

MODEL_CFG = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

train_samples = len(DatasetCatalog.get(DATASET_TRAIN))
val_samples = len(DatasetCatalog.get(DATASET_VAL))

batch_size = 1
iters_per_epoch = max(1, math.ceil(train_samples / batch_size))

target_epochs = 260
max_iter = min(36000, max(18000, target_epochs * iters_per_epoch))
step1 = int(max_iter * 0.70)
step2 = int(max_iter * 0.88)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(MODEL_CFG))

cfg.DATASETS.TRAIN = (DATASET_TRAIN,)
cfg.DATASETS.TEST = (DATASET_VAL,)

# No worker subprocesses: avoids duplicate host-RAM usage.
cfg.DATALOADER.NUM_WORKERS = 0

cfg.MODEL.DEVICE = DEVICE
cfg.OUTPUT_DIR = os.path.join(OUT_ROOT, "detectron2_output")
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(MODEL_CFG)

# ------------------------------
# Solver
# ------------------------------
cfg.SOLVER.IMS_PER_BATCH = 1
cfg.SOLVER.BASE_LR = 1.0e-4
cfg.SOLVER.MAX_ITER = max_iter
cfg.SOLVER.STEPS = (step1, step2)
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.WARMUP_ITERS = min(1000, max(300, int(0.05 * max_iter)))
cfg.SOLVER.CHECKPOINT_PERIOD = max(1000, iters_per_epoch * 10)

# Mixed precision on T4 reduces activation memory.
cfg.SOLVER.AMP.ENABLED = True

cfg.SOLVER.CLIP_GRADIENTS.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0
cfg.SOLVER.CLIP_GRADIENTS.NORM_TYPE = 2.0

# Less frequent full COCO evaluation to avoid RAM spikes.
cfg.TEST.EVAL_PERIOD = max(3000, iters_per_epoch * 25)

# ------------------------------
# Dense/small-object tuning — memory-safe
# ------------------------------
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[4, 8, 16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]

# The important V2 change:
# sharply reduce proposals participating in pairwise IoU matching.
cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.RPN.NMS_THRESH = 0.7
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 512
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1500
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 500

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.35

# Prevent very large instance objects during validation/inference.
cfg.TEST.DETECTIONS_PER_IMAGE = 500

# Smaller full-image validation size.
cfg.INPUT.MIN_SIZE_TEST = 512
cfg.INPUT.MAX_SIZE_TEST = 768

# These are also used by components that inspect cfg input settings.
cfg.INPUT.MIN_SIZE_TRAIN = (512,)
cfg.INPUT.MAX_SIZE_TRAIN = 768

BEST_METRIC = "segm/AP50"
BEST_WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_best_ap50.pth")


class GammaPrimeTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        output_folder = output_folder or os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(
            dataset_name,
            output_dir=output_folder,
            max_dets_per_image=cfg.TEST.DETECTIONS_PER_IMAGE,
        )

    @classmethod
    def build_train_loader(cls, cfg):
        # Random 512×512 crops are the key protection for very dense
        # micrographs: fewer GT instances are simultaneously involved in
        # proposal/GT IoU matching, which prevents the CUDA→CPU OOM cascade.
        mapper = DatasetMapper(
            cfg,
            is_train=True,
            augmentations=[
                T.RandomCrop("absolute", (512, 512)),
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
                T.RandomFlip(prob=0.5, horizontal=False, vertical=True),
                T.RandomRotation(angle=[0, 90, 180, 270], sample_style="choice"),
                T.RandomBrightness(0.9, 1.1),
                T.RandomContrast(0.9, 1.1),
            ],
        )
        return build_detection_train_loader(
            cfg,
            mapper=mapper,
            num_workers=0,
        )

    def build_hooks(self):
        hooks_list = super().build_hooks()
        hooks_list.insert(
            -1,
            hooks.BestCheckpointer(
                self.cfg.TEST.EVAL_PERIOD,
                DetectionCheckpointer(self.model, self.cfg.OUTPUT_DIR),
                BEST_METRIC,
                mode="max",
                file_prefix="model_best_ap50",
            ),
        )
        return hooks_list


print(f"Train images: {train_samples} | Val images: {val_samples}")
print(f"Iterations/epoch: {iters_per_epoch}")
print(f"MAX_ITER: {max_iter} | STEPS: {(step1, step2)}")
print("Backbone:", MODEL_CFG)
print("Training crop: 512 x 512")
print("Workers:", cfg.DATALOADER.NUM_WORKERS)
print("RPN PRE/POST train:", cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN,
      cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN)
print("ROI batch/image:", cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE)
print("Validation max size:", cfg.INPUT.MAX_SIZE_TEST)
print("Detections/image:", cfg.TEST.DETECTIONS_PER_IMAGE)
print("Best checkpoint:", BEST_WEIGHTS)

gc.collect()
torch.cuda.empty_cache()

trainer = GammaPrimeTrainer(cfg)
trainer.resume_or_load(resume=False)


In [ ]:
# ============================================================
# TRAIN — V2 memory-safe
# ============================================================
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU before training:")
print(torch.cuda.get_device_name(0))
print("Allocated GB:", round(torch.cuda.memory_allocated()/1024**3, 3))
print("Reserved  GB:", round(torch.cuda.memory_reserved()/1024**3, 3))

trainer.train()

print("Training finished. Output:", cfg.OUTPUT_DIR)
print("Final weights:", os.path.join(cfg.OUTPUT_DIR, "model_final.pth"))

gc.collect()
torch.cuda.empty_cache()


# **Evaluate on VAL (COCO AP) and save results JSON**

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

FINAL_WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
print("Evaluating weights:", FINAL_WEIGHTS)

# Make sure the trainer model uses the chosen weights
DetectionCheckpointer(trainer.model).load(FINAL_WEIGHTS)

#evaluator = COCOEvaluator(DATASET_VAL, output_dir=cfg.OUTPUT_DIR)
evaluator = COCOEvaluator(
    DATASET_VAL,
    output_dir=cfg.OUTPUT_DIR,
    max_dets_per_image=500
)
val_loader = build_detection_test_loader(cfg, DATASET_VAL)

results = inference_on_dataset(trainer.model, val_loader, evaluator)
print(results)

# Save as json for figures
res_path = os.path.join(OUT_ROOT, "coco_eval_results.json")
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)
print("Saved:", res_path)

summary = {
    "final_weights_used": FINAL_WEIGHTS,
    "best_metric": BEST_METRIC,
    "segm_AP": results.get("segm", {}).get("AP", None),
    "segm_AP50": results.get("segm", {}).get("AP50", None),
    "segm_AP75": results.get("segm", {}).get("AP75", None),
}
with open(os.path.join(OUT_ROOT, "evaluation_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved:", os.path.join(OUT_ROOT, "evaluation_summary.json"))


# **COCO AP50/AP75 bar chart**

In [ ]:
import matplotlib.pyplot as plt

with open(os.path.join(OUT_ROOT, "coco_eval_results.json"), "r") as f:
    R = json.load(f)

ap50 = R["segm"]["AP50"]
ap75 = R["segm"]["AP75"]
ap   = R["segm"]["AP"]

vals = [ap, ap50, ap75]
labels = ["AP(50:95)", "AP50", "AP75"]

plt.figure(figsize=(6,4))
x = np.arange(len(vals))
plt.bar(x, vals)
plt.xticks(x, labels)
plt.ylabel("Segmentation AP")
plt.title("Mask R-CNN (γ′) — COCO Segmentation AP")
for i,v in enumerate(vals):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()

save_path = os.path.join(FIG_DIR, "fig_coco_ap_bars.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

# **Prediction overlays (VAL)**

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer, ColorMode

cfg_pred = cfg.clone()
cfg_pred.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg_pred.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.20
predictor = DefaultPredictor(cfg_pred)

def show_pred_examples(dataset_name, n=6, save_path=None):
    dset = DatasetCatalog.get(dataset_name)
    meta = MetadataCatalog.get(dataset_name)

    picks = random.sample(dset, min(n, len(dset)))

    cols = 3
    rows = int(np.ceil(len(picks)/cols))
    plt.figure(figsize=(5*cols, 5*rows))

    for i, rec in enumerate(picks):
        img_path = rec["file_name"]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        out = predictor(img_rgb)
        inst = out["instances"].to("cpu")

        v = Visualizer(img_rgb, metadata=meta, scale=1.0, instance_mode=ColorMode.IMAGE)
        vis = v.draw_instance_predictions(inst)

        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(vis.get_image())
        ax.set_title(f"{os.path.basename(img_path)} | pred inst={len(inst)}")
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()

pred_save = os.path.join(FIG_DIR, "fig_pred_overlays_val.png")
show_pred_examples(DATASET_VAL, n=6, save_path=pred_save)


# **Threshold sweep (0.05 -> 0.5): mean Dice + mean instance count**

In [ ]:
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from skimage.morphology import remove_small_objects

coco = COCO(SINGLE_VAL_JSON)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygons
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def gt_union_mask(img_id):
    img = coco.loadImgs(img_id)[0]
    H, W = img["height"], img["width"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))
    return union, img["file_name"]

def dice(a, b, eps=1e-7):
    a = (a > 0).astype(np.uint8)
    b = (b > 0).astype(np.uint8)
    inter = (a & b).sum()
    return (2*inter + eps) / (a.sum() + b.sum() + eps)

def apply_union_postprocess(mask01, min_size=16):
    mask01 = (mask01 > 0)
    if min_size and min_size > 1:
        mask01 = remove_small_objects(mask01, min_size=min_size)
    return mask01.astype(np.uint8)

img_ids = coco.getImgIds()
print("VAL images:", len(img_ids))

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
rows = []

for thr in thresholds:
    cfg_thr = cfg.clone()
    cfg_thr.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    cfg_thr.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(thr)
    predictor_thr = DefaultPredictor(cfg_thr)

    dices = []
    inst_counts = []

    for img_id in img_ids:
        gt, fname = gt_union_mask(img_id)
        img_path = os.path.join(IMG_DIR, fname)
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        out = predictor_thr(img_rgb)
        inst = out["instances"].to("cpu")
        inst_counts.append(int(len(inst)))

        if len(inst) == 0:
            pred_union = np.zeros_like(gt, dtype=np.uint8)
        else:
            pm = inst.pred_masks.numpy().astype(np.uint8)  # [N,H,W]
            pred_union = (np.any(pm, axis=0)).astype(np.uint8)

        pred_union = apply_union_postprocess(pred_union, min_size=16)
        dices.append(dice(gt, pred_union))

    rows.append({
        "score_thresh": thr,
        "mean_dice": float(np.mean(dices)),
        "std_dice": float(np.std(dices)),
        "mean_pred_inst": float(np.mean(inst_counts)),
    })

df_thr = pd.DataFrame(rows).sort_values(["mean_dice", "score_thresh"], ascending=[False, True]).reset_index(drop=True)
print(df_thr)

best_row = df_thr.iloc[0].to_dict()
BEST_SCORE_THRESH = float(best_row["score_thresh"])
print("Selected BEST_SCORE_THRESH:", BEST_SCORE_THRESH)

# plot
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(df_thr["score_thresh"], df_thr["mean_dice"], marker="o")
plt.xlabel("Score threshold")
plt.ylabel("Mean Dice (union mask)")
plt.title("Threshold sweep — Dice vs score threshold")
plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig_threshold_sweep_dice.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

df_thr.to_csv(os.path.join(OUT_ROOT, "threshold_sweep.csv"), index=False)
with open(os.path.join(OUT_ROOT, "threshold_sweep_best.json"), "w") as f:
    json.dump({"BEST_SCORE_THRESH": BEST_SCORE_THRESH, "best_row": best_row}, f, indent=2)
print("Saved CSV:", os.path.join(OUT_ROOT, "threshold_sweep.csv"))
print("Saved JSON:", os.path.join(OUT_ROOT, "threshold_sweep_best.json"))


# **Deployment: run on ALL images -> save masks + overlays + CSV**

In [ ]:
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

def compute_shape_metrics_from_union(mask01):
    mask01 = (mask01 > 0).astype(np.uint8)

    area_frac = mask01.mean()

    cc = label(mask01)
    props = regionprops(cc)

    if len(props) == 0:
        return {
            "gamma_prime_area_fraction": float(area_frac),
            "gamma_prime_cc_count": 0,
            "gamma_prime_mean_eq_diameter_px": 0.0,
            "gamma_prime_mean_aspect_ratio": 0.0,
            "gamma_prime_mean_circularity": 0.0,
        }

    eqd = []
    ar  = []
    circ = []

    for p in props:
        eqd.append(p.equivalent_diameter)
        maj = p.major_axis_length if p.major_axis_length > 1e-9 else np.nan
        mino = p.minor_axis_length if p.minor_axis_length > 1e-9 else np.nan
        ar.append(float(maj/mino) if (mino and not np.isnan(mino) and not np.isnan(maj)) else np.nan)

        per = p.perimeter if p.perimeter > 1e-9 else np.nan
        a = p.area
        circ.append(float(4*np.pi*a/(per*per)) if (per and not np.isnan(per)) else np.nan)

    eqd = np.array(eqd, dtype=float)
    ar  = np.array(ar, dtype=float)
    circ= np.array(circ, dtype=float)

    return {
        "gamma_prime_area_fraction": float(area_frac),
        "gamma_prime_cc_count": int(len(props)),
        "gamma_prime_mean_eq_diameter_px": float(np.nanmean(eqd)),
        "gamma_prime_mean_aspect_ratio": float(np.nanmean(ar)),
        "gamma_prime_mean_circularity": float(np.nanmean(circ)),
    }

def overlay_pred_mask(img_rgb, mask01, fill_alpha=0.35, edge_thickness=2):
    # create colored overlay in green
    img = img_rgb.copy()
    H,W = mask01.shape
    m = (mask01 > 0)

    # fill
    color = np.array([0, 255, 0], dtype=np.uint8)
    img_f = img.astype(np.float32)
    img_f[m] = img_f[m]*(1-fill_alpha) + color*(fill_alpha)

    out = np.clip(img_f, 0, 255).astype(np.uint8)

    # edge
    edges = cv2.Canny((mask01*255).astype(np.uint8), 50, 150)
    ys, xs = np.where(edges > 0)
    for y,x in zip(ys,xs):
        y0 = max(0, y-edge_thickness); y1 = min(H, y+edge_thickness+1)
        x0 = max(0, x-edge_thickness); x1 = min(W, x+edge_thickness+1)
        out[y0:y1, x0:x1] = [255, 255, 0]  # yellow boundary
    return out

# predictor for deployment
cfg_deploy = cfg.clone()
cfg_deploy.MODEL.WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg_deploy.MODEL.ROI_HEADS.SCORE_THRESH_TEST = BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20
deploy_pred = DefaultPredictor(cfg_deploy)

# run on all images in IMG_DIR
image_paths = []
for fn in sorted(os.listdir(IMG_DIR)):
    if fn.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff")):
        image_paths.append(os.path.join(IMG_DIR, fn))

print("Images found:", len(image_paths))
print("Deployment threshold:", cfg_deploy.MODEL.ROI_HEADS.SCORE_THRESH_TEST)

rows = []
for img_path in image_paths:
    fname = os.path.basename(img_path)

    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    out = deploy_pred(img_rgb)
    inst = out["instances"].to("cpu")

    if len(inst) == 0:
        union01 = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    else:
        pm = inst.pred_masks.numpy().astype(np.uint8)
        union01 = (np.any(pm, axis=0)).astype(np.uint8)

    # Light post-processing to suppress tiny noise
    union01 = remove_small_objects(union01.astype(bool), min_size=16).astype(np.uint8)

    m = compute_shape_metrics_from_union(union01)

    mask_path = os.path.join(PRED_DIR, fname.rsplit(".",1)[0] + "_pred_mask.png")
    save_mask = (union01*255).astype(np.uint8)
    cv2.imwrite(mask_path, save_mask)

    ov = overlay_pred_mask(img_rgb, union01)
    ov_path = os.path.join(OVR_DIR, fname.rsplit(".",1)[0] + "_overlay.png")
    cv2.imwrite(ov_path, cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

    rows.append({
        "file": fname,
        "pred_mask_path": mask_path,
        "overlay_path": ov_path,
        **m
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(OUT_ROOT, "deployment_predictions_and_metrics.csv")
df.to_csv(csv_out, index=False)
print("Saved:", csv_out)
df.head()


# **Distributions of extracted microstructure metrics**

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(OUT_ROOT, "deployment_predictions_and_metrics.csv"))

plots = [
    ("gamma_prime_area_fraction", "Area fraction (γ′)"),
    ("gamma_prime_cc_count", "Connected components (γ′ union)"),
    ("gamma_prime_mean_eq_diameter_px", "Mean equiv. diameter (px)"),
    ("gamma_prime_mean_aspect_ratio", "Mean aspect ratio"),
    ("gamma_prime_mean_circularity", "Mean circularity"),
]

for col, title in plots:
    plt.figure(figsize=(6,4))
    plt.hist(df[col].values, bins=25)
    plt.xlabel(title)
    plt.ylabel("Count")
    plt.title(f"Distribution: {title}")
    plt.tight_layout()

    sp = os.path.join(FIG_DIR, f"fig_hist_{col}.png")
    plt.savefig(sp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", sp)

# **Saved overlay images**

In [ ]:
import matplotlib.pyplot as plt

overlay_files = [os.path.join(OVR_DIR, f) for f in sorted(os.listdir(OVR_DIR)) if f.lower().endswith(".png")]
print("Overlays:", len(overlay_files))

# choose a good spread
k = min(12, len(overlay_files))
picks = np.linspace(0, len(overlay_files)-1, k, dtype=int)
picks = [overlay_files[i] for i in picks]

cols = 4
rows = int(np.ceil(k/cols))
plt.figure(figsize=(4.5*cols, 4.5*rows))

for i, p in enumerate(picks):
    img = cv2.imread(p)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax = plt.subplot(rows, cols, i+1)
    ax.imshow(img)
    ax.set_title(os.path.basename(p).replace("_overlay.png",""))
    ax.axis("off")

plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig_montage_deployment_overlays.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

# **Setup helpers (Ground Truth union mask + error overlay)**

In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# This mUST match our earlier paths (IMG_DIR, SINGLE_VAL_JSON, FIG_DIR, cfg_pred, predictor)
print("IMG_DIR:", IMG_DIR)
print("SINGLE_VAL_JSON:", SINGLE_VAL_JSON)
print("FIG_DIR:", FIG_DIR)

coco_val = COCO(SINGLE_VAL_JSON)

def ann_to_mask(ann, H, W):
    seg = ann["segmentation"]
    if isinstance(seg, list):  # polygon list
        rles = maskUtils.frPyObjects(seg, H, W)
        rle = maskUtils.merge(rles)
    else:  # RLE
        rle = seg
    m = maskUtils.decode(rle)
    if m.ndim == 3:
        m = np.any(m, axis=2).astype(np.uint8)
    return (m > 0).astype(np.uint8)

def gt_union_mask_from_coco(img_id):
    img = coco_val.loadImgs(img_id)[0]
    H, W = img["height"], img["width"]
    fname = img["file_name"]
    ann_ids = coco_val.getAnnIds(imgIds=[img_id])
    anns = coco_val.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        union = np.maximum(union, ann_to_mask(ann, H, W))

    return union, fname, len(anns)

def pred_union_mask(img_rgb, predictor):
    out = predictor(img_rgb)
    inst = out["instances"].to("cpu")
    if len(inst) == 0:
        return np.zeros(img_rgb.shape[:2], dtype=np.uint8), 0
    pm = inst.pred_masks.numpy().astype(np.uint8)  # [N,H,W]
    union = (np.any(pm, axis=0)).astype(np.uint8)
    return union, int(len(inst))

def edges_from_mask(mask01, thickness=2):
    m = (mask01 > 0).astype(np.uint8) * 255
    e = cv2.Canny(m, 50, 150)
    if thickness > 1:
        k = np.ones((thickness, thickness), np.uint8)
        e = cv2.dilate(e, k, iterations=1)
    return (e > 0)

def draw_outline(img_rgb, mask01, color=(0,255,0), thickness=2):
    img = img_rgb.copy()
    e = edges_from_mask(mask01, thickness=thickness)
    img[e] = np.array(color, dtype=np.uint8)
    return img

def error_map_overlay(img_rgb, gt01, pr01, alpha=0.45):
    """
    TP: green, FN: red, FP: blue
    """
    gt = (gt01 > 0)
    pr = (pr01 > 0)

    tp = gt & pr
    fn = gt & (~pr)
    fp = (~gt) & pr

    overlay = img_rgb.copy().astype(np.float32)

    # colors
    green = np.array([0,255,0], np.float32)
    red   = np.array([255,0,0], np.float32)
    blue  = np.array([0,0,255], np.float32)

    for mask, col in [(tp, green), (fn, red), (fp, blue)]:
        overlay[mask] = overlay[mask]*(1-alpha) + col*alpha

    return np.clip(overlay, 0, 255).astype(np.uint8)

import json
import zipfile
import shutil


# **Ground Truth / Pred / Error grid**

In [ ]:
import random

def make_gt_pred_error_grid(n_images=6, seed=42, score_thresh=BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20, save_name="fig_gt_pred_error_grid.png"):
    random.seed(seed)

    # ensure predictor threshold
    cfg_pred.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(score_thresh)
    from detectron2.engine import DefaultPredictor
    predictor_local = DefaultPredictor(cfg_pred)

    img_ids = coco_val.getImgIds()
    picks = random.sample(img_ids, min(n_images, len(img_ids)))

    rows = len(picks)
    cols = 3

    plt.figure(figsize=(4.2*cols, 4.2*rows))

    for r, img_id in enumerate(picks):
        gt01, fname, gt_count = gt_union_mask_from_coco(img_id)
        img_path = os.path.join(IMG_DIR, fname)

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print("[WARN] cannot read:", img_path)
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        pr01, pred_count = pred_union_mask(img_rgb, predictor_local)

        # panels
        gt_panel   = draw_outline(img_rgb, gt01, color=(0,255,0), thickness=2)     # green
        pred_panel = draw_outline(img_rgb, pr01, color=(255,255,0), thickness=2)   # yellow
        err_panel  = error_map_overlay(img_rgb, gt01, pr01, alpha=0.45)

        # plot GT
        ax = plt.subplot(rows, cols, r*cols + 1)
        ax.imshow(gt_panel)
        ax.set_title(f"{fname} | GT (inst={gt_count})", fontsize=10)
        ax.axis("off")

        # plot Pred
        ax = plt.subplot(rows, cols, r*cols + 2)
        ax.imshow(pred_panel)
        ax.set_title(f"{fname} | Pred (inst={pred_count})", fontsize=10)
        ax.axis("off")

        # plot Error
        ax = plt.subplot(rows, cols, r*cols + 3)
        ax.imshow(err_panel)
        ax.set_title(f"{fname} | Error (TP=G, FN=R, FP=B)", fontsize=10)
        ax.axis("off")

    plt.tight_layout()

    save_path = os.path.join(FIG_DIR, save_name)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

make_gt_pred_error_grid(
    n_images=6,          # we can change this to 9 or 12
    seed=7,
    score_thresh=BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20,
    save_name="fig_gt_pred_error_grid.png"
)

# **To check if we are using the right COCO + image folder**

In [ ]:

import os
from pycocotools.coco import COCO

COCO_JSON = COCO_ALL_JSON
IMG_DIR   = IMG_DIR

print("COCO_JSON:", COCO_JSON)
print("IMG_DIR  :", IMG_DIR, "| exists:", os.path.isdir(IMG_DIR))

coco = COCO(COCO_JSON)

print("COCO images:", len(coco.dataset["images"]))
print("COCO anns  :", len(coco.dataset["annotations"]))
print("COCO cats  :", coco.dataset["categories"])

# show 10 example filenames + whether they exist in IMG_DIR
for img in coco.dataset["images"][:10]:
    p = os.path.join(IMG_DIR, img["file_name"])
    print(img["file_name"], " | exists:", os.path.exists(p))


# **Select 5 images that have Ground Truth and exist on drive**

In [ ]:
import random

def gt_count_for_image(img_id):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    return len(ann_ids)

candidates = []
for img in coco.dataset["images"]:
    img_id = img["id"]
    fname = img["file_name"]
    img_path = os.path.join(IMG_DIR, fname)
    if os.path.exists(img_path):
        n = gt_count_for_image(img_id)
        if n > 0:
            candidates.append((img_id, fname, n))

print("Candidates (exists on disk + GT>0):", len(candidates))
print("First 10 candidates:", candidates[:10])

selected = random.sample(candidates, 5) if len(candidates) >= 5 else candidates
print("\nSelected:")
for img_id, fname, n in selected:
    print(fname, "| GT anns:", n)

# **COCO mask decode (works for polygon + RLE)**

In [ ]:
import numpy as np
from pycocotools import mask as maskUtils

def ann_to_mask(ann, height, width):
    seg = ann.get("segmentation", None)
    if seg is None:
        return np.zeros((height, width), dtype=np.uint8)

    # Polygon
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, height, width)
        rle = maskUtils.merge(rles)
        m = maskUtils.decode(rle)

    # RLE dict
    elif isinstance(seg, dict) and "counts" in seg:
        rle = seg
        m = maskUtils.decode(rle)

    else:
        return np.zeros((height, width), dtype=np.uint8)

    if m.ndim == 3:
        m = m[..., 0]
    return (m > 0).astype(np.uint8)

# **(Original | GT union | GT overlay)**

In [ ]:
import cv2, matplotlib.pyplot as plt

def overlay_fill_and_edge(img_rgb, mask01, fill_alpha=0.35):
    """
    img_rgb: HxWx3 uint8
    mask01:  HxW uint8 (0/1)
    """
    img = img_rgb.copy().astype(np.float32)

    # Fill (green)
    fill = np.array([0, 255, 0], dtype=np.float32)
    m = (mask01 > 0)
    img[m] = img[m] * (1 - fill_alpha) + fill * fill_alpha

    # Edge (white) from contours
    edges = np.zeros(mask01.shape, dtype=np.uint8)
    cnts, _ = cv2.findContours(mask01.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(edges, cnts, -1, 255, 1)
    img[edges > 0] = np.array([255, 255, 255], dtype=np.float32)

    return np.clip(img, 0, 255).astype(np.uint8)

def make_triptych(img_id, fname):
    img_info = coco.loadImgs([img_id])[0]
    H, W = img_info["height"], img_info["width"]
    img_path = os.path.join(IMG_DIR, fname)

    bgr = cv2.imread(img_path)
    if bgr is None:
        print("[WARN] Cannot read:", img_path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    nonempty = 0
    for ann in anns:
        m = ann_to_mask(ann, H, W)
        if m.sum() > 0:
            nonempty += 1
        union = np.maximum(union, m)

    overlay = overlay_fill_and_edge(rgb, union, fill_alpha=0.35)

    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1); plt.imshow(rgb); plt.title(f"Original\n{fname}")
    plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(union, cmap="gray"); plt.title(f"GT union mask\nanns={len(anns)} | decoded_nonempty={nonempty}")
    plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(overlay); plt.title("GT overlay (fill+edge)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Show 5 images

In [ ]:
for img_id, fname, n in selected:
    make_triptych(img_id, fname)

In [ ]:

SAVE_DIR = os.path.join(OUT_ROOT, "gt_visualisations")
os.makedirs(SAVE_DIR, exist_ok=True)

def save_triptych(img_id, fname):
    img_info = coco.loadImgs([img_id])[0]
    H, W = img_info["height"], img_info["width"]
    img_path = os.path.join(IMG_DIR, fname)

    bgr = cv2.imread(img_path)
    if bgr is None:
        print("[WARN] Cannot read:", img_path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    union = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        m = ann_to_mask(ann, H, W)
        union = np.maximum(union, m)

    overlay = overlay_fill_and_edge(rgb, union, fill_alpha=0.35)
    base = os.path.splitext(fname)[0]

    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_original.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_mask.png"), union * 255)
    cv2.imwrite(os.path.join(SAVE_DIR, f"{base}_overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    fig = plt.figure(figsize=(14, 4))
    plt.subplot(1, 3, 1); plt.imshow(rgb); plt.axis("off")
    plt.subplot(1, 3, 2); plt.imshow(union, cmap="gray"); plt.axis("off")
    plt.subplot(1, 3, 3); plt.imshow(overlay); plt.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, f"{base}_triptych.png"))
    plt.close(fig)

    print(f"[SAVED] {base} → {SAVE_DIR}")

for img_id, fname, n in selected:
    save_triptych(img_id, fname)


# **COCO evaluation bar chart (AP / AP50 / AP75)**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert isinstance(results, dict), "Expected `results` to be a dict from Detectron2 evaluation."
assert "bbox" in results or "segm" in results, "Expected results to contain 'bbox' and/or 'segm'."

OUT_DIR = FIG_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# Build tidy dataframe

rows = []
for task in ["bbox", "segm"]:
    if task not in results:
        continue
    for metric in ["AP", "AP50", "AP75"]:
        if metric in results[task]:
            rows.append({"task": task, "metric": metric, "value": float(results[task][metric])})

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError("No metrics found in results. Check results keys and metric names.")

# Keep nice ordering
metric_order = ["AP", "AP50", "AP75"]
task_order = [t for t in ["bbox", "segm"] if t in df["task"].unique()]

# Pivot for plotting
pivot = (
    df.pivot(index="metric", columns="task", values="value")
      .reindex(metric_order)
      .loc[metric_order]
)

# Plot + display
x = np.arange(len(pivot.index))
width = 0.35

plt.figure(figsize=(8, 4.8))

for j, task in enumerate(task_order):
    vals = pivot[task].values
    plt.bar(x + (j - (len(task_order)-1)/2)*width, vals, width=width, label=task)

plt.xticks(x, pivot.index)
plt.ylabel("COCO metric value")
plt.title("COCO evaluation summary (1-class γ′)")
plt.legend()
plt.tight_layout()

# Save plot
plot_path = os.path.join(OUT_DIR, "coco_eval_summary_ap_ap50_ap75.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

display(df)

csv_path = os.path.join(OUT_DIR, "coco_eval_summary_table.csv")
df.to_csv(csv_path, index=False)
print("Saved table to:", csv_path)

# **DEPLOYMENT**

# Install/Imports (run once)

In [ ]:
# To be skipped if detectron2 is already installed.
# !pip install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git'
!pip -q install gradio opencv-python-headless pycocotools

import os, cv2, json, math
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

import torch
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.model_zoo import get_config_file
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model

# **Set paths + build predictor**

In [ ]:
# Detectron2 config name (Mask R-CNN)
CFG_NAME = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

# Trained weights (.pth) – prefer the best validation checkpoint
WEIGHTS = BEST_WEIGHTS if os.path.exists(BEST_WEIGHTS) else os.path.join(cfg.OUTPUT_DIR, "model_final.pth")

# Image folder to run deployment on
DEPLOY_IMG_DIR = IMG_DIR

# Where to save outputs
OUT_DIR = OUT_ROOT
PRED_MASK_DIR = os.path.join(OUT_DIR, "pred_masks")
OVERLAY_DIR   = os.path.join(OUT_DIR, "overlays")
CSV_PATH      = os.path.join(OUT_DIR, "deployment_predictions_and_metrics.csv")
BATCH_EXPORT_ROOT = os.path.join(OUT_DIR, "batch_exports")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(BATCH_EXPORT_ROOT, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SCORE_THRESH = BEST_SCORE_THRESH if "BEST_SCORE_THRESH" in globals() else 0.20
MIN_COMPONENT_SIZE = 16

from detectron2.config import get_cfg
from detectron2.model_zoo import get_config_file
from detectron2.engine import DefaultPredictor

cfg = get_cfg()
cfg.merge_from_file(get_config_file(CFG_NAME))

cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[4, 8, 16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]

cfg.MODEL.RPN.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.RPN.NMS_THRESH = 0.7
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST  = 1500
cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 500

cfg.TEST.DETECTIONS_PER_IMAGE = 500
cfg.INPUT.MIN_SIZE_TEST = 512
cfg.INPUT.MAX_SIZE_TEST = 768

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.35
cfg.MODEL.WEIGHTS = WEIGHTS
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH
cfg.MODEL.DEVICE = DEVICE

predictor = DefaultPredictor(cfg)

print("Device:", DEVICE)
print("Weights exists:", os.path.exists(WEIGHTS))
print("Saving to:", OUT_DIR)
print("Batch CVAT exports root:", BATCH_EXPORT_ROOT)
print("Deployment SCORE_THRESH:", SCORE_THRESH)
print("MIN_COMPONENT_SIZE:", MIN_COMPONENT_SIZE)


# **Helper functions (overlay + measurements)**

In [ ]:
def postprocess_instance_masks(masks_bool, min_area=MIN_COMPONENT_SIZE if "MIN_COMPONENT_SIZE" in globals() else 16):
    cleaned = []
    for m in masks_bool:
        m2 = remove_small_objects(m.astype(bool), min_size=min_area)
        if np.any(m2):
            cleaned.append(m2.astype(bool))
    return cleaned


def read_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def save_mask_png(mask01, save_path):
    # mask01: uint8 0/1 or bool
    m = (mask01.astype(np.uint8) * 255)
    cv2.imwrite(save_path, m)

def overlay_fill_and_boundary(img_rgb, mask01, fill_color=(0,255,0), edge_color=(255,255,255),
                              fill_alpha=0.35, edge_thickness=2):
    """
    img_rgb: HxWx3 uint8
    mask01: HxW (bool or 0/1)
    """
    img = img_rgb.copy()
    m = mask01.astype(bool)

    # Fill
    if m.any():
        color = np.array(fill_color, dtype=np.float32)
        img_f = img.astype(np.float32)
        img_f[m] = img_f[m] * (1 - fill_alpha) + color * fill_alpha
        img = np.clip(img_f, 0, 255).astype(np.uint8)

    # Boundary via contours
    m8 = (mask01.astype(np.uint8) * 255)
    cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) > 0:
        cv2.drawContours(img, cnts, -1, edge_color, thickness=edge_thickness)

    return img

def compute_instance_metrics_from_masks(masks_bool):
    """
    masks_bool: list of HxW bool arrays (one per instance)
    returns dict of aggregate stats
    """
    if len(masks_bool) == 0:
        return dict(
            pred_instance_count=0,
            area_fraction=0.0,
            cc_count=0,
            mean_area_px=np.nan,
            mean_eq_diameter_px=np.nan,
            mean_aspect_ratio=np.nan,
            mean_circularity=np.nan,
            mean_perimeter_px=np.nan,
            mean_nn_spacing_px=np.nan,
        )

    H, W = masks_bool[0].shape
    union = np.zeros((H, W), dtype=np.uint8)
    areas = []
    perims = []
    eqds = []
    ars = []
    circs = []
    centroids = []

    for m in masks_bool:
        m8 = (m.astype(np.uint8) * 255)
        union = np.maximum(union, m8)

        area = float(m.sum())
        areas.append(area)

        cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(cnts) == 0:
            perims.append(0.0)
            eqds.append(0.0)
            ars.append(np.nan)
            circs.append(np.nan)
            centroids.append((np.nan, np.nan))
            continue

        cnt = max(cnts, key=cv2.contourArea)
        perim = float(cv2.arcLength(cnt, True))
        perims.append(perim)

        # Equivalent diameter: sqrt(4A/pi)
        eqd = math.sqrt(4.0 * area / math.pi) if area > 0 else 0.0
        eqds.append(eqd)

        # Aspect ratio: from min-area rectangle
        rect = cv2.minAreaRect(cnt)  # ((cx,cy),(w,h),theta)
        (cx, cy), (rw, rh), _ = rect
        if rw > 0 and rh > 0:
            ar = max(rw, rh) / min(rw, rh)
        else:
            ar = np.nan
        ars.append(ar)

        # Circularity: 4πA / P^2
        circ = (4.0 * math.pi * area / (perim**2)) if perim > 1e-6 else np.nan
        circs.append(circ)

        centroids.append((float(cx), float(cy)))

    # Connected components on UNION mask
    union01 = (union > 0).astype(np.uint8)
    num_cc, _ = cv2.connectedComponents(union01)
    cc_count = int(max(0, num_cc - 1))

    area_fraction = float(union01.sum()) / float(H * W)

    # Nearest-neighbour spacing (centroid-to-centroid)
    pts = np.array([[x, y] for (x, y) in centroids if np.isfinite(x) and np.isfinite(y)], dtype=np.float32)
    if len(pts) >= 2:
        dists = []
        for i in range(len(pts)):
            diff = pts - pts[i]
            dd = np.sqrt((diff[:,0]**2 + diff[:,1]**2))
            dd[i] = np.inf
            dists.append(np.min(dd))
        mean_nn = float(np.mean(dists))
    else:
        mean_nn = np.nan

    return dict(
        pred_instance_count=int(len(masks_bool)),
        area_fraction=float(area_fraction),
        cc_count=cc_count,
        mean_area_px=float(np.nanmean(areas)),
        mean_eq_diameter_px=float(np.nanmean(eqds)),
        mean_aspect_ratio=float(np.nanmean(ars)),
        mean_circularity=float(np.nanmean(circs)),
        mean_perimeter_px=float(np.nanmean(perims)),
        mean_nn_spacing_px=float(mean_nn),
    )

def mask_to_coco_segmentation(mask_bool, approx_epsilon=1.5):
    """Convert a binary instance mask to COCO polygon segmentation."""
    m8 = (mask_bool.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    segmentation = []
    for cnt in contours:
        if approx_epsilon and approx_epsilon > 0:
            cnt = cv2.approxPolyDP(cnt, approx_epsilon, True)
        if cnt.shape[0] < 3:
            continue
        poly = cnt.reshape(-1, 2).astype(float).flatten().tolist()
        if len(poly) >= 6:
            segmentation.append(poly)
    return segmentation

def build_coco_annotation(mask_bool, image_id, ann_id, category_id=1):
    segmentation = mask_to_coco_segmentation(mask_bool)
    if len(segmentation) == 0:
        return None
    ys, xs = np.where(mask_bool)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    bbox = [float(x0), float(y0), float(x1 - x0 + 1), float(y1 - y0 + 1)]
    area = float(mask_bool.sum())
    return {
        "id": int(ann_id),
        "image_id": int(image_id),
        "category_id": int(category_id),
        "segmentation": segmentation,
        "area": area,
        "bbox": bbox,
        "iscrowd": 0,
    }

def make_run_root(base_root=BATCH_EXPORT_ROOT, prefix="batch_cvat_export"):
    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    run_root = os.path.join(base_root, f"{prefix}_{ts}")
    os.makedirs(run_root, exist_ok=True)
    return run_root

def prepare_export_dirs(run_root):
    dirs = {
        "run_root": run_root,
        "images_dir": os.path.join(run_root, "images"),
        "masks_dir": os.path.join(run_root, "pred_masks"),
        "overlays_dir": os.path.join(run_root, "overlays"),
        "annotations_dir": os.path.join(run_root, "annotations"),
    }
    for p in dirs.values():
        os.makedirs(p, exist_ok=True)
    return dirs

def zip_dir(src_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(src_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, src_dir)
                zf.write(fpath, arcname)
    return zip_path

def infer_one_image(img_path, save_outputs=True, export_dirs=None, image_id=None, ann_start_id=1, copy_image=False):
    """
    Runs Detectron2 inference, saves mask+overlay, returns metrics dict + paths.
    If export_dirs is provided, also prepares per-instance COCO annotations for CVAT import.
    """
    img_rgb = read_rgb(img_path)
    H, W = img_rgb.shape[:2]

    outputs = predictor(img_rgb)
    inst = outputs["instances"].to("cpu")

    masks = []
    if inst.has("pred_masks"):
        pm = inst.pred_masks.numpy()  # [N,H,W] bool
        for i in range(pm.shape[0]):
            masks.append(pm[i].astype(bool))

    masks = postprocess_instance_masks(masks)

    # Union mask
    if len(masks) > 0:
        union = np.zeros((H, W), dtype=np.uint8)
        for m in masks:
            union = np.maximum(union, m.astype(np.uint8))
        union01 = union.astype(np.uint8)
    else:
        union01 = np.zeros((H, W), dtype=np.uint8)

    fname = os.path.basename(img_path)
    base = os.path.splitext(fname)[0]
    if export_dirs is None:
        mask_path = os.path.join(PRED_MASK_DIR, f"{base}_pred_mask.png")
        overlay_path = os.path.join(OVERLAY_DIR, f"{base}_overlay.png")
        image_copy_path = img_path
    else:
        mask_path = os.path.join(export_dirs["masks_dir"], f"{base}_pred_mask.png")
        overlay_path = os.path.join(export_dirs["overlays_dir"], f"{base}_overlay.png")
        image_copy_path = os.path.join(export_dirs["images_dir"], fname)

    if save_outputs:
        save_mask_png(union01, mask_path)
        ov = overlay_fill_and_boundary(img_rgb, union01, fill_color=(0,255,0), edge_color=(255,255,255))
        cv2.imwrite(overlay_path, cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))
        if export_dirs is not None and copy_image:
            shutil.copy2(img_path, image_copy_path)
    else:
        mask_path = ""
        overlay_path = ""
        image_copy_path = img_path

    m = compute_instance_metrics_from_masks(masks)
    row = dict(
        file=fname,
        image_path=image_copy_path,
        source_image_path=img_path,
        pred_mask_path=mask_path,
        overlay_path=overlay_path,
        **m
    )

    image_record = None
    annotations = []
    next_ann_id = ann_start_id
    if export_dirs is not None and image_id is not None:
        image_record = {
            "id": int(image_id),
            "file_name": fname,
            "width": int(W),
            "height": int(H),
        }
        for mask_bool in masks:
            ann = build_coco_annotation(mask_bool, image_id=image_id, ann_id=next_ann_id, category_id=1)
            if ann is not None:
                annotations.append(ann)
                next_ann_id += 1

    return row, image_record, annotations, next_ann_id


# **Run deployment on a folder (and export CSV)**

In [ ]:

def deploy_folder(img_dir, csv_path=CSV_PATH, export_cvat=False, run_root=None):
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"):
        image_paths += glob(os.path.join(img_dir, ext))
    image_paths = sorted(image_paths)

    if len(image_paths) == 0:
        raise ValueError(f"No images found in: {img_dir}")

    export_dirs = None
    zip_path = None
    annotations_json_path = None
    if export_cvat:
        if run_root is None:
            run_root = make_run_root()
        export_dirs = prepare_export_dirs(run_root)
        csv_path = os.path.join(run_root, "deployment_predictions_and_metrics.csv")

    rows = []
    coco_images = []
    coco_annotations = []
    ann_id = 1

    for image_id, p in enumerate(tqdm(image_paths, desc="Running deployment"), start=1):
        try:
            row, image_record, anns, ann_id = infer_one_image(
                p,
                save_outputs=True,
                export_dirs=export_dirs,
                image_id=image_id if export_cvat else None,
                ann_start_id=ann_id,
                copy_image=export_cvat,
            )
            rows.append(row)
            if export_cvat:
                coco_images.append(image_record)
                coco_annotations.extend(anns)
        except Exception as e:
            rows.append(dict(
                file=os.path.basename(p),
                image_path=p,
                source_image_path=p,
                pred_mask_path="",
                overlay_path="",
                pred_instance_count=0,
                area_fraction=np.nan,
                cc_count=np.nan,
                mean_area_px=np.nan,
                mean_eq_diameter_px=np.nan,
                mean_aspect_ratio=np.nan,
                mean_circularity=np.nan,
                mean_perimeter_px=np.nan,
                mean_nn_spacing_px=np.nan,
                error=str(e)
            ))

    df = pd.DataFrame(rows)

    if export_cvat:
        df.to_csv(csv_path, index=False)
        coco = {
            "info": {
                "description": "Predicted gamma_prime masks for CVAT correction",
                "version": "1.0",
            },
            "licenses": [],
            "images": coco_images,
            "annotations": coco_annotations,
            "categories": [{"id": 1, "name": "gamma_prime", "supercategory": "microstructure"}],
        }
        annotations_json_path = os.path.join(export_dirs["annotations_dir"], "instances_default.json")
        with open(annotations_json_path, "w") as f:
            json.dump(coco, f)

        readme_path = os.path.join(run_root, "README_for_CVAT.txt")
        with open(readme_path, "w") as f:
            f.write(
                "1. In CVAT, create a segmentation task and upload the same images from the images folder.\n"
                "2. Then use Actions -> Upload annotations.\n"
                "3. Select format COCO 1.0 and upload annotations/instances_default.json.\n"
                "4. The pred_masks folder contains union masks for quick inspection; CVAT uses the COCO JSON.\n"
            )

        zip_path = os.path.join(run_root, "cvat_predictions_bundle.zip")
        zip_dir(run_root, zip_path)
        print("Saved CVAT bundle:", zip_path)
    else:
        if os.path.exists(csv_path):
            old = pd.read_csv(csv_path)
            df_all = pd.concat([old, df], ignore_index=True)
            df_all.to_csv(csv_path, index=False)
        else:
            df.to_csv(csv_path, index=False)

    print("Saved:", csv_path)
    return df, csv_path, zip_path, run_root, annotations_json_path

df_results, latest_csv_path, latest_zip_path, latest_run_root, latest_annotations_json = deploy_folder(
    DEPLOY_IMG_DIR,
    csv_path=CSV_PATH,
    export_cvat=False,
)
df_results.head()


# **Quick visualization to show 5 examples (Original | Mask | Overlay)**

In [ ]:
import matplotlib.pyplot as plt

def show_examples(df, n=5, seed=42):
    df2 = df.sample(n=min(n, len(df)), random_state=seed)
    for _, r in df2.iterrows():
        img = read_rgb(r["image_path"])
        m = cv2.imread(r["pred_mask_path"], cv2.IMREAD_GRAYSCALE) if os.path.exists(r["pred_mask_path"]) else None
        ov = cv2.imread(r["overlay_path"], cv2.IMREAD_COLOR) if os.path.exists(r["overlay_path"]) else None
        if ov is not None:
            ov = cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(14,4))
        plt.subplot(1,3,1); plt.title(f"Original: {r['file']}"); plt.imshow(img); plt.axis("off")
        plt.subplot(1,3,2); plt.title("Pred union mask"); plt.imshow(m, cmap="gray"); plt.axis("off")
        plt.subplot(1,3,3); plt.title("Overlay (fill + boundary)"); plt.imshow(ov); plt.axis("off")
        plt.tight_layout()
        plt.show()

show_examples(df_results, n=5)

# **Gradio Web App (single image + batch folder)**

In [ ]:
import gradio as gr

def gr_infer_single(img):
    """
    img is a numpy array RGB from gradio
    """
    if img is None:
        return None, None, pd.DataFrame()

    img_rgb = img.astype(np.uint8)
    H, W = img_rgb.shape[:2]

    outputs = predictor(img_rgb)
    inst = outputs["instances"].to("cpu")

    masks = []
    if inst.has("pred_masks"):
        pm = inst.pred_masks.numpy()
        for i in range(pm.shape[0]):
            masks.append(pm[i].astype(bool))

    if len(masks) > 0:
        union01 = np.zeros((H, W), dtype=np.uint8)
        for m in masks:
            union01 = np.maximum(union01, m.astype(np.uint8))
    else:
        union01 = np.zeros((H, W), dtype=np.uint8)

    overlay = overlay_fill_and_boundary(img_rgb, union01, fill_color=(0,255,0), edge_color=(255,255,255))
    m = compute_instance_metrics_from_masks(masks)
    df = pd.DataFrame([m])
    mask_vis = (union01 * 255).astype(np.uint8)
    return overlay, mask_vis, df

def gr_deploy_folder(img_dir):
    df, csv_path, zip_path, run_root, annotations_json_path = deploy_folder(
        img_dir,
        csv_path=CSV_PATH,
        export_cvat=True,
        run_root=None,
    )
    status = (
        f"Done. Export folder: {run_root}\n"
        f"CSV: {csv_path}\n"
        f"COCO JSON: {annotations_json_path}\n"
        f"CVAT ZIP bundle: {zip_path}"
    )
    return df, csv_path, zip_path, status

with gr.Blocks() as demo:
    gr.Markdown(
        """## γ′ Deployment + Measurement Pipeline (Mask R-CNN)

Upload an image or run a whole folder batch.

Batch export saves:
- overlay
- union masks
- CSV
- images
- COCO JSON
- CVAT-ready ZIP bundle
"""
    )

    with gr.Tab("Single Image"):
        inp = gr.Image(type="numpy", label="Upload micrograph")
        btn = gr.Button("Run Inference + Measurements")
        out_overlay = gr.Image(type="numpy", label="Overlay (fill + boundary)")
        out_mask = gr.Image(type="numpy", label="Predicted union mask (PNG-style)")
        out_table = gr.Dataframe(label="Measurements (per image)")

        btn.click(fn=gr_infer_single, inputs=[inp], outputs=[out_overlay, out_mask, out_table])

    with gr.Tab("Batch Folder"):
        folder_in = gr.Textbox(value=IMG_DIR, label="Folder path containing images")
        btn2 = gr.Button("Run Batch Deployment + Save CSV + Export CVAT Bundle")
        out_df = gr.Dataframe(label="Batch Results (preview)")
        out_csv = gr.File(label="Download CSV")
        out_zip = gr.File(label="Download CVAT ZIP bundle")
        out_status = gr.Textbox(label="Export status", lines=5)

        btn2.click(fn=gr_deploy_folder, inputs=[folder_in], outputs=[out_df, out_csv, out_zip, out_status])

demo.launch(share=True, debug=True)